In [5]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/NL_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()


             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  00000000000000000000_0 -0.259853       1  0.584752  0.320851  0.630442   
1  00000000000000000000_0 -0.198258       1  0.445639  0.268564  0.505145   
2  00000000000000000000_0 -0.173547       1  0.411047  0.244099  0.459005   
3  00000000000000000000_0 -0.175744       1  0.418317  0.241056  0.485535   
4  00000000000000000001_0 -0.279493       1  0.622885  0.353801  0.616120   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  94.475369   
1  2022  {"geodesic":false,"type":"Point","coordinates"...  94.475369   
2  2023  {"geodesic":false,"type":"Point","coordinates"...  94.475369   
3  2024  {"geodesic":false,"type":"Point","coordinates"...  94.475369   
4  2021  {"geodesic":false,"type":"Point","coordinates"...  94.476268   

    latitude  
0  26.507937  
1  26.507937  
2  26.507937  
3  26.507937  
4  26.507937  
[2021 20

In [7]:
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)

Raw input shape: (30000, 4, 5)


In [8]:

# ===============================
# 4. Label Creation (BEFORE SCALING)
# ===============================
# NDVI drop = vegetation loss
ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # 2021 - 2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # 2024 - 2021

ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# ===============================
# 5. Train–Test Split (WITH LOCATIONS)
# ===============================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
    X_raw, y, locations,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# ===============================
# 6. Scaling (NO DATA LEAKAGE)
# ===============================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# ===============================
# 7. LSTM Model
# ===============================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision')
    ]
)

model.summary()

# ===============================
# 8. Train Model (Class Weighting)
# ===============================
class_weight = {0: 1, 1: 12}

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    class_weight=class_weight,
    verbose=1
)

# ===============================
# 9. Evaluation
# ===============================
from sklearn.metrics import classification_report

y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)   # recall-friendly threshold

print(classification_report(y_test, y_pred))

# ===============================
# 10. Save Predictions (CORRECT LOCATIONS)
# ===============================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/NL_Deforestation_Predictions.csv',
    index=False
)

print("Saved:NL_Deforestation_Predictions.csv")



Label distribution:
No deforestation: 26022
Deforestation: 3978
Train shape: (24000, 4, 5)
Test shape : (6000, 4, 5)
Scaled train shape: (24000, 4, 5)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.2512 - loss: 1.4797 - precision: 0.1462 - recall: 0.9472 - val_accuracy: 0.7617 - val_loss: 0.5385 - val_precision: 0.3445 - val_recall: 0.8819
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.7779 - loss: 0.8644 - precision: 0.3566 - recall: 0.8770 - val_accuracy: 0.8220 - val_loss: 0.4058 - val_precision: 0.4221 - val_recall: 0.9259
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.8168 - loss: 0.7119 - precision: 0.4074 - recall: 0.9049 - val_accuracy: 0.9270 - val_loss: 0.2000 - val_precision: 0.6815 - val_recall: 0.8442
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8681 - loss: 0.5809 - precision: 0.4965 - recall: 0.9309 - val_accuracy: 0.8377 - val_loss: 0.3759 - val_precision: 0.4490 - val_recall: 0.9837
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8595 - loss: 0.5391 - precision: 0.4873 - recall: 0.9447 - val_accuracy: 0.8063 - val_loss

In [9]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/NL_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/NL_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    # Missing data protection
    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    # 🔥 Fire → extreme burn signal
    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    # 🪓 Logging → vegetation drop + soil exposure
    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    # 🌾 Agriculture → moderate change
    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    # 🏙 Urban / other disturbance
    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/NL_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


NDVI: {'mean': np.float64(-0.25726930972533335), 'std': 0.06983294151004828, 'q1': np.float64(-0.29499536000000004), 'q3': np.float64(-0.2282973), 'lower_2std': np.float64(-0.3969351927454299), 'upper_2std': np.float64(-0.1176034267052368)}
NBR : {'mean': np.float64(-0.187771744764805), 'std': 0.08213641321605784, 'q1': np.float64(-0.21938240750000004), 'q3': np.float64(-0.16788638500000003), 'lower_2std': np.float64(-0.3520445711969207), 'upper_2std': np.float64(-0.0234989183326893)}
BSI : {'mean': np.float64(0.09617075048776959), 'std': 0.0591211481586775, 'q1': np.float64(0.07704229340088414), 'q3': np.float64(0.12361086578667987), 'lower_2std': np.float64(-0.022071545829585412), 'upper_2std': np.float64(0.21441304680512457)}
NDMI: {'mean': np.float64(-0.07666557119202816), 'std': 0.06105623944174454, 'q1': np.float64(-0.10330222749999998), 'q3': np.float64(-0.05469472749999999), 'lower_2std': np.float64(-0.19877805007551724), 'upper_2std': np.float64(0.045446907691460925)}

Defores

In [13]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# Andhra Pradesh map
# AP roughly: latitude 12.6–19.9, longitude 76.8–84.8
m = folium.Map(location=[16.5, 80.6], zoom_start=7)

# Load CSV
df = pd.read_csv('/content/drive/MyDrive/NL_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}
# Cluster points for better performance
marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)


cause
Urban/Other    1294
Unknown         178
Agriculture      11
Name: count, dtype: int64


In [14]:
m.save('/content/drive/MyDrive/nl_adaptive.html')


In [18]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================

df_def = pd.read_csv(
    '/content/drive/MyDrive/NL_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()

print("Total deforestation samples:", len(df_deforest))


# ==============================
# LOAD CAUSE DATA
# ==============================

df_cause = pd.read_csv(
    '/content/drive/MyDrive/NL_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())


# ==============================
# LOAD NAGALAND DISTRICTS
# ==============================

nl = gpd.read_file('/content/drive/MyDrive/NL.geojson')

# CRS SAFETY
if nl.crs is None:
    nl.set_crs("EPSG:4326", inplace=True)

nl = nl.to_crs("EPSG:4326")

# Add state label (optional)
nl["state"] = "Nagaland"

print(nl.head())


# ==============================
# MERGE CAUSE DATA
# ==============================

df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())


# ==============================
# CREATE POINT GEOMETRY
# ==============================

geometry = [
    Point(xy) for xy in zip(
        df_deforest['longitude'],
        df_deforest['latitude']
    )
]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 6000
    latitude  longitude  deforestation
0  26.529496  94.698151              0
1  26.531293  94.662219              0
2  26.539378  95.088919              0
3  26.517818  95.051189              1
4  26.543869  94.821221              1
Total deforestation samples: 1483
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  26.517818  95.051189              1    -0.234425   -0.147290    0.033998   
1  26.543869  94.821221              1    -0.206270   -0.072096    0.030554   
2  26.552852  94.826611              1    -0.219658   -0.099844    0.023176   
3  26.549259  94.763728              1    -0.205577   -0.160426    0.054704   
4  26.525005  94.464589              1    -0.231705   -0.175799    0.075931   

   NDMI_change        cause  
0    -0.044050  Urban/Other  
1    -0.023760  Urban/Other  
2    -0.007577  Urban/Other  
3    -0.043499  Urban/Other  
4    -0.043425  Urban/Other  
  REMARKS_2 Country State_Name State_Code   Dist_Name Dis

In [19]:
# ==============================
# SPATIAL JOIN (Points → Districts)
# ==============================

gdf_joined = gpd.sjoin(
    gdf_points,
    nl,                # Nagaland GeoDataFrame
    how='left',
    predicate='intersects'
)

print(gdf_joined.head())

print("Total joined points:", len(gdf_joined))
print("Points without district:", gdf_joined['Dist_Name'].isna().sum())

print(gdf_joined['Dist_Name'].value_counts())


# ==============================
# FIX DUPLICATE CAUSE COLUMNS
# ==============================

if 'cause' not in gdf_joined.columns:

    if 'cause_x' in gdf_joined.columns and 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x'].fillna(gdf_joined['cause_y'])

    elif 'cause_x' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_x']

    elif 'cause_y' in gdf_joined.columns:
        gdf_joined['cause'] = gdf_joined['cause_y']

# cleanup
gdf_joined.drop(
    columns=[c for c in ['cause_x','cause_y'] if c in gdf_joined.columns],
    inplace=True
)


# ==============================
# DISTRICT SUMMARY — NL
# ==============================

district_summary = (
    gdf_joined
    .groupby('Dist_Name')
    .size()
    .reset_index(name='deforestation_points')
    .sort_values(by='deforestation_points', ascending=False)
)

print(district_summary)

print(
    "Sum of district-wise deforestation samples:",
    district_summary['deforestation_points'].sum()
)

district_summary.to_csv(
    '/content/drive/MyDrive/NL_District_Wise_Deforestation.csv',
    index=False
)

print("Saved NL district summary")


# ==============================
# DISTRICT × CAUSE SUMMARY
# ==============================

cause_summary = (
    gdf_joined
    .groupby(['Dist_Name', 'cause'])
    .size()
    .reset_index(name='count')
)

print(cause_summary.head())

cause_summary.to_csv(
    '/content/drive/MyDrive/NL_District_Wise_Deforestation_Causes.csv',
    index=False
)

print("Saved NL cause summary")

    latitude  longitude  deforestation        cause  \
0  26.517818  95.051189              1  Urban/Other   
1  26.543869  94.821221              1  Urban/Other   
2  26.552852  94.826611              1  Urban/Other   
3  26.549259  94.763728              1  Urban/Other   
4  26.525005  94.464589              1  Urban/Other   

                    geometry  index_right REMARKS_2 Country State_Name  \
0  POINT (95.05119 26.51782)            3      None   India   Nagaland   
1  POINT (94.82122 26.54387)            9      None   India   Nagaland   
2  POINT (94.82661 26.55285)            9      None   India   Nagaland   
3  POINT (94.76373 26.54926)            9      None   India   Nagaland   
4    POINT (94.46459 26.525)            2      None   India   Nagaland   

  State_Code   Dist_Name Dist_Code     state  
0         13         Mon       261  Nagaland  
1         13    Longleng       268  Nagaland  
2         13    Longleng       268  Nagaland  
3         13    Longleng       268  

In [20]:
# =====================================================
# STEP 1: Imports
# =====================================================
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 2: Load Nagaland District Boundaries
# =====================================================
nl = gpd.read_file('/content/drive/MyDrive/NL.geojson')

# CRS safety
if nl.crs is None:
    nl.set_crs(epsg=4326, inplace=True)

nl = nl.to_crs(epsg=4326)
nl["state"] = "Nagaland"
gdf_districts = nl.copy()

# =====================================================
# STEP 3: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/NL_Deforestation_Predictions.csv")

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 4: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 5: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 6: District Statistics
# =====================================================
district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 7: Merge Stats
# =====================================================
gdf_districts = gdf_districts.merge(district_total, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_deforestation, on="Dist_Name", how="left")
gdf_districts = gdf_districts.merge(district_afforestation, on="Dist_Name", how="left")

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 8: Area + Rate Calculations
# =====================================================
PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 9: Create Folium Map (center Nagaland)
# =====================================================
m = folium.Map(location=[26.1, 94.3], zoom_start=8)

# =====================================================
# STEP 10: Choropleth
# =====================================================
min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Nagaland)"
).add_to(m)

# =====================================================
# STEP 11: Tooltips
# =====================================================
folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 12: Nagaland Afforestation Alert Popup
# =====================================================
state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(by="deforestation_samples", ascending=False)
    .head(3)
)

top_districts_html = ""
for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Nagaland Afforestation Alert</b><br><br>

Total Deforestation Points: <b>{int(state_def)}</b><br><br>

High Impact Districts:<br>
{top_districts_html}<br>

🌱 Immediate afforestation drives recommended.<br>
Focus on community forestry, biodiversity conservation,<br>
and sustainable land practices.
"""

folium.Marker(
    location=[26.1, 94.3],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 13: Save Map
# =====================================================
folium.LayerControl().add_to(m)
m.save("/content/drive/MyDrive/nl_forest.html")

print("✅ Nagaland map saved successfully with afforestation recommendation!")

/tmp/ipython-input-470/2255310988.py:82: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Nagaland map saved successfully with afforestation recommendation!
